# CipherMark — le couple embedder / extracteur

**Mémoire de Master 2 · Université de Yaoundé I**
*Faculté des Sciences — Département d'Informatique*

Ce carnet fait une seule chose, de bout en bout : **écrire** un champ témoin Ω
dans une image, puis le **relire**. Rien d'autre. Le banc d'essai complet, qui
couvre les décodeurs génératifs et la chaîne prompt→verdict, est dans
`CipherMark-banc-essai-des-modeles.ipynb`.

---

## Ce que vous pouvez faire ici

| section | ce qu'elle montre |
|---|---|
| 2 | charger un des couples entraînés, et voir ce qu'il a coûté en fidélité |
| 3 | tatouer votre image, comparer avant/après, voir le résidu amplifié |
| 4 | relire Ω et obtenir un verdict avec sa p-valeur |
| 5 | **le contrôle négatif** — une image non tatouée, une mauvaise clé |
| 6 | attaquer l'image (JPEG, bruit, flou, recadrage) et revérifier |
| 7 | la courbe complète force du filigrane ↔ fidélité ↔ lisibilité |

## Avant de commencer

**Exécution → Modifier le type d'exécution → GPU.** Le hash perceptuel
utilise DINOv2 ; sans GPU tout fonctionne mais lentement.

Les modèles sont lus depuis votre Drive, dans `ciphermark/`. La première
cellule le monte et installe les dépendances : comptez trois minutes.

In [ ]:
#@title  1 · Mise en route — Drive, dépendances, chargeurs  { display-mode: "form" }
#@markdown Monte le Drive, récupère le code, installe ce qui manque et définit
#@markdown les fonctions utilisées par les cellules suivantes. À relancer après
#@markdown chaque redémarrage de session.

import os, sys, shutil, subprocess, time
t0 = time.time()

RACINE_DRIVE = "/content/drive/MyDrive/ciphermark"
DEPOT        = "/content/code-memoire"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("hors Colab : on suppose le depot et les checkpoints deja locaux")

# --- le code -----------------------------------------------------------------
# L'archive du Drive contient un dossier racine (code-memoire/) : la
# decompresser dans /content/code-memoire donnerait un niveau de trop, et
# "No module named distseal.utils". On extrait donc a cote, puis on localise
# le dossier qui contient reellement distseal/ au lieu de le supposer.
if not os.path.isdir(os.path.join(DEPOT, "distseal")):
    arch = os.path.join(RACINE_DRIVE, "00-code-source-a-jour.tar.gz")
    if os.path.exists(arch):
        tmp = "/content/_extraction"
        shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp, exist_ok=True)
        subprocess.run(["tar", "xzf", arch, "-C", tmp], check=True)
        racine = next((r for r, _, _ in os.walk(tmp)
                       if os.path.isdir(os.path.join(r, "distseal", "utils"))), None)
        if racine is None:
            raise SystemExit("l'archive ne contient pas distseal/utils")
        shutil.rmtree(DEPOT, ignore_errors=True)
        shutil.move(racine, DEPOT)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", "ciphermark",
                        "https://github.com/ngueagho/distseal-meta.git", DEPOT], check=True)

if not os.path.isdir(os.path.join(DEPOT, "distseal", "utils")):
    raise SystemExit(f"{DEPOT} ne contient pas distseal/utils -- recuperation du code echouee")
sys.path[:0] = [os.path.join(DEPOT, "deps"), DEPOT]
os.chdir(DEPOT)
print(f"   code : {DEPOT}  (distseal/utils present)")

# --- les dependances, resolues par essais successifs -------------------------
# La liste de ce qui manque varie d'une session Colab a l'autre ; la deviner
# coute des relances ratees. On tente l'import, on lit le module absent dans
# l'exception, on installe, on recommence.
PIP = {"cv2": "opencv-python-headless", "PIL": "pillow", "skimage": "scikit-image"}
import importlib
for _ in range(15):
    try:
        importlib.import_module("distseal.utils.cfg")
        break
    except ModuleNotFoundError as e:
        m = e.name.split(".")[0]
        print("   installation de", PIP.get(m, m))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIP.get(m, m)])

# Certaines dependances ne se signalent PAS par un ModuleNotFoundError : phash.py
# rattrape l'absence de reedsolo dans un try et la transforme en RuntimeError au
# moment de l'usage. La boucle ci-dessus ne peut donc pas la voir, et l'erreur
# n'apparaitrait qu'a la section 3, apres trois minutes d'installation. On les
# verifie explicitement.
# reedsolo : rattrape dans phash.py, echoue a l'usage en section 3.
# Crypto (pycryptodome) : crypto.py bascule sinon sur un repli HMAC-CTR qui
#   produit un keystream DIFFERENT -- les marques posees ailleurs deviendraient
#   inverifiables ici, en silence. C'est le piege le plus grave des trois.
for mod, paquet in (("reedsolo", "reedsolo"), ("Crypto", "pycryptodome"),
                    ("huggingface_hub", "huggingface_hub"),
                    ("matplotlib", "matplotlib"),
                    ("skimage", "scikit-image"), ("timm", "timm")):
    if importlib.util.find_spec(mod) is None:
        print("   installation de", paquet)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet])

import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("\n   ATTENTION : pas de GPU. Execution -> Modifier le type d'execution -> GPU\n")

# --- catalogue des couples entraines -----------------------------------------
# (chemin sur le Drive, ce que le modele a coute et rapporte)
COUPLES = {
 "phase H — fidelite (37,8 dB, RECOMMANDE)":
   ("06-entrainements-en-cours-non-termines/phaseH-tatoueur-fidelite-attenuation-jnd/checkpoint.pth",
    "Affine pour l'invisibilite : 37,85 dB a bit_acc 0,9991. 15 dB de mieux "
    "que la reference pour 0,0007 de precision en moins."),
 "reference 64 bits (22,5 dB)":
   ("01-tatoueur-modeles-de-reference-64-et-128-bits/phaseA2_64bits_stable_checkpoint.pth",
    "Le couple de reference, bit_acc 0,9998. C'est lui qui sert de professeur "
    "partout ailleurs dans le projet. Tatouage visible a l'oeil."),
 "phase B — robuste aux attaques":
   ("03-tatoueur-robuste-aux-attaques-passives/checkpoint.pth",
    "Entraine avec augmentations : porte le flou k7 de 72,5 a 98,2 % de "
    "verdicts, et recupere 23,2 % des images sous recadrage a 90 %."),
 "128 bits (canal plus large)":
   ("01-tatoueur-modeles-de-reference-64-et-128-bits/phaseA2_128bits_stable_checkpoint.pth",
    "Deux fois plus de bits, bit_acc 0,9616 : la lisibilite se paie. A 256 "
    "bits le canal sature et plus aucun verdict n'est rendu -- c'est le mur "
    "de capacite documente au chapitre 3."),
}

def _copie(rel):
    src, loc = os.path.join(RACINE_DRIVE, rel), "/content/ckpt/" + rel.replace("/", "__")
    os.makedirs("/content/ckpt", exist_ok=True)
    if not os.path.exists(loc):
        if not os.path.exists(src):
            raise SystemExit(f"absent du Drive : {src}")
        print(f"   copie de {os.path.getsize(src)/2**20:.0f} Mo depuis le Drive...")
        shutil.copy(src, loc)
    return loc

def charger_couple(nom, force=None, bavard=True, user_id=""):
    """Charge l'embedder, l'extracteur, le hash et la chaine de verification.

    user_id : si renseigne, la cle du temoin est DERIVEE de cet identifiant,
    k_user = HKDF-Expand(k_master, "ciphermark/v1/user" || user_id). Deux
    utilisateurs sur la meme image produisent alors des Omega differents, et
    l'operateur peut regenerer la cle de n'importe lequel pour instruire une
    attribution contestee. Laisse vide, les cles sont anonymes : la marque
    prouve qu'une image vient d'un detenteur de cle, mais pas DE QUI.
    """
    global WAM, CM, PHASH, CLES, REGISTRE, NBITS, IMG_SIZE, TAU, HASH_BITS, FORCE, USER_ID
    from distseal.utils.cfg import get_config_from_checkpoint, setup_model_from_checkpoint
    from distseal.utils import optim as uoptim
    from distseal.ciphermark.phash import PerceptualHash, _try_load_dinov2, _DCTFallback
    from distseal.ciphermark.registry import TraceRegistry
    from distseal.ciphermark.wam_ciphermark import CipherMarkConfig, CipherMarkKeys, CipherMarkWam

    rel, note = COUPLES[nom]
    loc = _copie(rel)
    cfg = get_config_from_checkpoint(loc)
    WAM = setup_model_from_checkpoint(loc).to(DEVICE).eval()
    NBITS, IMG_SIZE = int(cfg.args.nbits), int(cfg.args.img_size)

    # Le calendrier de force du filigrane DOIT etre rejoue. Sans cela on
    # applique l'amplitude du DEBUT d'entrainement -- 0,5 au lieu de 0,1152 --
    # et l'image sort rayee, avec 8 dB de PSNR au lieu de 22.
    if getattr(cfg.args, "scaling_w_schedule", None):
        ck = torch.load(loc, map_location="cpu", weights_only=True)
        if ck.get("epoch") is not None:
            pr = uoptim.parse_params(cfg.args.scaling_w_schedule)
            uoptim.ScalingScheduler(obj=WAM.blender, attribute="scaling_w",
                                    scaling_o=cfg.args.scaling_w, **pr).step(ck["epoch"])
        del ck
    entraine = float(WAM.blender.scaling_w)
    FORCE = entraine if force is None else float(force)
    WAM.blender.scaling_w = FORCE

    HASH_BITS = 256
    dino = _try_load_dinov2()
    PHASH = PerceptualHash(n_bits=HASH_BITS, backbone=dino or _DCTFallback()).to(DEVICE).eval()
    CLES, REGISTRE = CipherMarkKeys.random(), TraceRegistry()
    CM = CipherMarkWam(wam=WAM, phash=PHASH, keys=CLES,
                       cfg=CipherMarkConfig(n_bits=NBITS, max_fixed_point_iters=3),
                       registry=REGISTRE)
    TAU = round(CM.cfg.hamming_threshold * HASH_BITS)

    USER_ID = (user_id or "").strip()
    if USER_ID:
        from distseal.ciphermark.witness import WitnessField
        from distseal.ciphermark.equation import CipherMarkVerifier
        CM.witness = WitnessField.for_user(s_master=CLES.s_master,
                                           k_master=CLES.k_secret,
                                           user_id=USER_ID, cfg=CM.witness.cfg)
        # Le verifieur est construit A PARTIR du witness : sans le reconstruire,
        # il garderait l'ancienne cle et rejetterait tout ce que l'embedder pose.
        CM.verifier = CipherMarkVerifier(CM.witness)

    if bavard:
        print(f"\n   {nom}")
        print(f"   {note}")
        print(f"\n   Omega {NBITS} bits | images {IMG_SIZE} px | {DEVICE}")
        print(f"   force du filigrane : {entraine:.4f} (entrainee) -> {FORCE:.4f} (appliquee)")
        print(f"   hash : {'DINOv2-small' if dino is not None else 'repli DCT'}, "
              f"{HASH_BITS} bits, seuil tau = {TAU}")
        if USER_ID:
            print(f"   utilisateur : \"{USER_ID}\" -- la cle du temoin en est DERIVEE,")
            print( "                 l'attribution est donc opposable.")
        else:
            print("   utilisateur : aucun. Cles anonymes : la marque prouve qu'une")
            print("                 image vient d'un detenteur de cle, mais pas DE QUI.")
        print("   La cle maitresse est tiree au hasard a chaque chargement.")

def image_de_travail(source="exemple (skimage)", taille=None, chemin=""):
    """Renvoie un tenseur (1,3,H,W) dans [0,1] et le nom de la source.

    chemin : si renseigne, prime sur `source` -- permet de designer une image
    deja presente sur le Drive ou dans /content sans passer par le televersement.
    """
    taille = taille or IMG_SIZE
    if chemin:
        chemin = chemin.strip()
        if not os.path.exists(chemin):
            raise SystemExit(f"introuvable : {chemin}\n"
                             f"   (un chemin du Drive ressemble a "
                             f"/content/drive/MyDrive/mes-images/photo.jpg)")
        im = Image.open(chemin).convert("RGB")
        a = np.asarray(im.resize((taille, taille))).astype(np.float32) / 255.
        return torch.from_numpy(a).permute(2, 0, 1)[None].to(DEVICE), os.path.basename(chemin)
    if source.startswith("televerser"):
        from google.colab import files
        envoi = files.upload()
        if not envoi: raise SystemExit("aucun fichier recu")
        nom = list(envoi)[0]
        import io as _io
        im = Image.open(_io.BytesIO(envoi[nom])).convert("RGB")
    else:
        from skimage import data
        im = Image.fromarray(data.chelsea()).convert("RGB"); nom = "chelsea (skimage)"
    a = np.asarray(im.resize((taille, taille))).astype(np.float32) / 255.
    return torch.from_numpy(a).permute(2, 0, 1)[None].to(DEVICE), nom

def psnr(a, b):
    m = float(((a - b) ** 2).mean())
    return float("inf") if m == 0 else 10 * np.log10(1.0 / m)

def montre(paires, titre=None, taille=3.4):
    """Affiche une rangee d'images, avec de l'air entre elles et au-dessus.

    tight_layout seul tasse les vignettes contre le titre et les unes contre
    les autres ; on reserve donc explicitement les marges.
    """
    n = len(paires)
    fig, ax = plt.subplots(1, n, figsize=(taille * n, taille + 0.9))
    if n == 1: ax = [ax]
    for a, (lg, img) in zip(ax, paires):
        a.imshow(img[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy())
        a.set_title(lg, fontsize=9, pad=8); a.axis("off")
    if titre:
        fig.suptitle(titre, fontsize=11, y=0.99)
    fig.subplots_adjust(wspace=0.12, top=0.84 if titre else 0.92,
                        bottom=0.03, left=0.02, right=0.98)
    plt.show()
    print()          # une ligne vide entre la figure et le texte qui suit


def separateur(titre=""):
    """Une respiration entre deux blocs de resultats."""
    print("\n" + "=" * 72)
    if titre:
        print(f"  {titre}")
        print("=" * 72)


plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "figure.constrained_layout.use": False})
print(f"\n   pret en {time.time()-t0:.0f} s — passez a la section 2")

---
# 2 · Choisir un couple

Quatre couples ont été entraînés, chacun réglant différemment le compromis
entre **invisibilité** et **lisibilité**. La phase H est celle à montrer : elle
gagne 15 dB de qualité d'image en ne perdant que 0,0007 de précision.

In [ ]:
#@title  2 · Charger le couple  { display-mode: "form" }
COUPLE = "phase H \u2014 fidelite (37,8 dB, RECOMMANDE)" #@param ["phase H \u2014 fidelite (37,8 dB, RECOMMANDE)", "reference 64 bits (22,5 dB)", "phase B \u2014 robuste aux attaques", "128 bits (canal plus large)"]
#@markdown ---
#@markdown **Force du filigrane** — laissez à 0 pour utiliser la valeur
#@markdown d'entraînement. La monter rend Ω plus lisible et l'image plus
#@markdown dégradée ; la baisser fait l'inverse. La section 7 trace la courbe.
FORCE = 0 #@param {type:"slider", min:0, max:0.16, step:0.005}
#@markdown ---
#@markdown **Identifiant du créateur** — laissez vide pour des clés anonymes.
#@markdown Renseigné, la clé du témoin en est *dérivée* : deux utilisateurs sur
#@markdown la même image produisent alors des Ω différents, et l'attribution
#@markdown devient opposable. Utilisez une identité synthétique.
USER_ID = "createur-0042" #@param {type:"string"}

if "charger_couple" not in globals():
    raise SystemExit(
        "\n  ==> Executez d'abord la cellule 1 (Mise en route), en haut du carnet.\n"
        "      Elle definit les chargeurs, et disparait a chaque redemarrage\n"
        "      de la session Colab.\n")
charger_couple(COUPLE, force=None if FORCE == 0 else FORCE,
               user_id=USER_ID)

---
# 3 · Écrire Ω dans une image

Le champ témoin n'est pas un message arbitraire : il vaut

$$\Omega = \mathrm{HMAC}(K_{\text{secret}},\, h) \oplus \mathrm{PRG}(s_{\text{master}},\, \text{nonce})$$

où $h$ est le hash perceptuel de **cette image**. Deux images différentes, ou
deux clés différentes, donnent deux Ω sans rapport.

La troisième vignette montre le résidu — la différence entre l'image marquée et
l'originale — **amplifié ×20**, sans quoi on ne verrait rien.

In [ ]:
#@title  3 · Tatouer  { display-mode: "form" }
SOURCE = "exemple (skimage)" #@param ["exemple (skimage)", "televerser une image", "chemin ci-dessous"]
#@markdown ---
#@markdown **Chemin d'une image** — utilisé seulement si vous choisissez
#@markdown « chemin ci-dessous ». Pratique pour une image déjà sur le Drive,
#@markdown par exemple `/content/drive/MyDrive/mes-images/photo.jpg`.
CHEMIN = "" #@param {type:"string"}

if "CM" not in globals():
    raise SystemExit("\n  ==> Chargez d'abord un couple (section 2).\n")

import time
x, origine = image_de_travail(SOURCE, chemin=CHEMIN if SOURCE.startswith("chemin") else "")
t0 = time.time()
sortie = CM.embed(x)
marquee, ids = sortie["imgs_w"], sortie["image_ids"]
dt = time.time() - t0

residu = (marquee - x)
montre([("originale", x),
        ("marquee", marquee),
        (f"residu x20 (max {float(residu.abs().max()):.3f})", residu.abs() * 20)],
       titre=f"{origine} — {dt:.1f} s")

separateur("RESULTAT DU TATOUAGE")
print(f"  PSNR      : {psnr(x, marquee):.2f} dB   (au-dela de 35, l'oeil ne voit rien)")
print(f"  nonce     : {ids[0]}   (l'identifiant de cette image dans le registre)")
print(f"  amplitude : residu max {float(residu.abs().max()):.4f} sur une echelle 0-1")
print("\n  L'image marquee reste en memoire pour les sections 4 a 6.")

---
# 4 · Relire Ω et rendre un verdict

Le vérificateur fait **deux** contrôles, et les deux doivent passer :

1. il extrait Ω de l'image et le compare au témoin attendu, recalculé depuis
   les clés — c'est la **distance de Hamming**, assortie d'une p-valeur ;
2. il recalcule le hash perceptuel de l'image observée et le compare à la
   référence enregistrée — c'est la **liaison au contenu**, qui empêche de
   recopier un Ω valide sur une autre image.

Un verdict `NOT_WATERMARKED` avec une distance d'Ω nulle n'est donc pas
incohérent : c'est le second contrôle qui a rejeté.

In [ ]:
#@title  4 · Extraire et verifier  { display-mode: "form" }
if "marquee" not in globals():
    raise SystemExit("\n  ==> Tatouez d'abord une image (section 3).\n")

separateur()
r = CM.verify(marquee, ids)[0]
h_obs = CM._phash_bytes(marquee)[0]
try:
    d_hash = CM._bit_distance(h_obs, CM.registry.h_ref_for(ids[0]))
except Exception:
    d_hash = None

print("=" * 66)
print(f"  VERDICT : {r.verdict.value.upper()}")
print("=" * 66)
print(f"  Omega   : {r.distance} bit(s) faux sur {r.total}"
      f"   (bit_acc {1 - r.distance/r.total:.4f})")
print(f"  p-valeur: {r.p_value:.3g}"
      f"   {'<< 1e-6, verdict rendu' if r.p_value < 1e-6 else '>= 1e-6, insuffisant'}")
if d_hash is not None:
    print(f"  contenu : hash a {d_hash} bits de la reference, seuil {TAU}"
          f"   -> {'DANS le rayon' if d_hash <= TAU else 'HORS du rayon'}")
print()
print("  La p-valeur est la probabilite qu'une image NON tatouee produise par")
print("  hasard une distance aussi faible. En dessous de 1e-6, le verdict est")
print("  rendu ; c'est ce seuil qui borne les fausses accusations.")

---
# 5 · Le contrôle négatif — ce qui rend le verdict crédible

Un système qui dirait toujours « authentique » passerait la section 4 sans
rien prouver. Deux contrôles l'excluent, et **doivent tous deux échouer** :

- une image **jamais tatouée** ne doit produire aucun verdict ;
- l'image marquée, vérifiée avec **une autre clé**, ne doit rien rendre non plus.

Les distances attendues sont d'environ la moitié des bits — le hasard pur.

In [ ]:
#@title  5 · Controle negatif  { display-mode: "form" }
if "marquee" not in globals():
    raise SystemExit("\n  ==> Tatouez d'abord une image (section 3).\n")

from distseal.ciphermark.wam_ciphermark import CipherMarkKeys, CipherMarkWam, CipherMarkConfig

separateur("CONTROLE NEGATIF — ces deux tests DOIVENT echouer")
print(f"  {'controle':<44}{'Omega':>10}{'p-valeur':>13}{'verdict':>10}")
print("  " + "-" * 76)

# 1. une image jamais tatouee, verifiee sous le meme nonce
r1 = CM.verify(x, ids)[0]
print(f"  {'image NON tatouee':<44}{str(r1.distance)+'/'+str(r1.total):>10}"
      f"{r1.p_value:>13.2g}{('RENDU' if r1.p_value < 1e-6 else 'aucun'):>10}")

# 2. l'image marquee, mais verifiee avec d'autres cles
autre = CipherMarkWam(wam=WAM, phash=PHASH, keys=CipherMarkKeys.random(),
                      cfg=CipherMarkConfig(n_bits=NBITS, max_fixed_point_iters=3),
                      registry=CM.registry)
r2 = autre.verify(marquee, ids)[0]
print(f"  {'image marquee, MAUVAISE cle':<44}{str(r2.distance)+'/'+str(r2.total):>10}"
      f"{r2.p_value:>13.2g}{('RENDU' if r2.p_value < 1e-6 else 'aucun'):>10}")

print()
ok = r1.p_value >= 1e-6 and r2.p_value >= 1e-6
print(f"  attendu : environ {NBITS//2}/{NBITS} bits faux, aucun verdict.")
print(f"  -> {'CONTROLES PASSES' if ok else 'ANOMALIE : un controle a rendu un verdict'}")
if ok:
    print("     Le verdict de la section 4 vient donc bien du temoin, et non")
    print("     d'un biais du detecteur qui dirait oui a tout.")

---
# 6 · Attaquer l'image, puis revérifier

Une image en ligne subit des transformations. La question n'est pas si Ω se
dégrade — il se dégrade toujours — mais si le **verdict** survit.

Chaque attaque donne sa propre figure : l'image **marquée** à gauche, l'image
**attaquée** à droite, et leur **différence** amplifiée, pour voir ce que
l'attaque a réellement détruit. Le bilan chiffré vient à la fin.

Le recadrage est la limite connue et déclarée du système : Ω est un message
global réparti sur toute l'image, non un motif répété localement. Le modèle
« phase B » est le seul entraîné avec augmentations et récupère une partie des
cas.

In [ ]:
#@title  6 · Attaques — une figure par attaque  { display-mode: "form" }
#@markdown Pour chaque transformation : l'image **marquée** à gauche, l'image
#@markdown **attaquée** à droite, et la **différence** entre les deux, amplifiée
#@markdown pour être visible. Le verdict est rappelé sous chaque figure.
AMPLIFICATION = 8 #@param {type:"slider", min:1, max:20, step:1}

if "marquee" not in globals():
    raise SystemExit("\n  ==> Tatouez d'abord une image (section 3).\n")

from distseal.augmentation import valuemetric, geometric
jpeg, flou, bruit = valuemetric.JPEG(), valuemetric.GaussianBlur(), valuemetric.GaussianNoise()
ATTAQUES = [
    ("JPEG Q=50",        lambda im: jpeg(im, None, quality=50)[0]),
    ("JPEG Q=30",        lambda im: jpeg(im, None, quality=30)[0]),
    ("bruit sigma=0,05", lambda im: bruit(im, None, std=0.05)[0]),
    ("flou k=7",         lambda im: flou(im, None, kernel_size=7)[0]),
    ("recadrage 70 %",   lambda im: geometric.Crop()(im, None, size=0.7)[0]),
]

# le point de comparaison : l'image marquee, non attaquee
r0 = CM.verify(marquee, ids)[0]
separateur("REFERENCE — l'image marquee, sans attaque")
print(f"  Omega {r0.distance}/{r0.total}   p = {r0.p_value:.2g}   "
      f"verdict {'RENDU' if r0.p_value < 1e-6 else 'aucun'}")

bilan = [("aucune", r0.distance, r0.p_value, float("inf"))]
for nom, f in ATTAQUES:
    separateur(f"ATTAQUE — {nom}")
    try:
        xa = f(marquee.clone()).clamp(0, 1)
        # le recadrage change la taille : on revient a celle du modele, comme
        # le ferait un verifieur reel devant une image rognee
        if xa.shape[-2:] != marquee.shape[-2:]:
            xa = F.interpolate(xa, size=marquee.shape[-2:], mode="bilinear",
                               align_corners=False, antialias=True)
            print("  (image ramenee a la taille du modele apres recadrage)")
        ra = CM.verify(xa, ids)[0]
        diff = (xa - marquee).abs()
        montre([("marquee (avant)", marquee),
                (f"apres {nom}", xa),
                (f"difference x{AMPLIFICATION}", diff * AMPLIFICATION)],
               titre=nom)
        print(f"  Omega    : {ra.distance}/{ra.total} bits faux"
              f"   (etait {r0.distance}/{r0.total} sans attaque)")
        print(f"  p-valeur : {ra.p_value:.2g}"
              f"   -> {'VERDICT RENDU' if ra.p_value < 1e-6 else 'aucun verdict'}")
        print(f"  degats   : PSNR {psnr(marquee, xa):.1f} dB entre avant et apres,"
              f" ecart max {float(diff.max()):.3f}")
        bilan.append((nom, ra.distance, ra.p_value, psnr(marquee, xa)))
    except Exception as e:
        print(f"  echec : {type(e).__name__}: {e}")

separateur("BILAN")
print(f"  {'attaque':<20}{'Omega':>10}{'p-valeur':>13}{'PSNR':>10}{'verdict':>10}")
print("  " + "-" * 63)
for nom, d, pv, ps in bilan:
    print(f"  {nom:<20}{str(d)+'/'+str(NBITS):>10}{pv:>13.2g}"
          f"{('inf' if ps == float('inf') else f'{ps:.1f}'):>10}"
          f"{('RENDU' if pv < 1e-6 else 'aucun'):>10}")
print()
print("  Le recadrage est la limite declaree du systeme : Omega est un message")
print("  global de 64 bits reparti sur toute l'image, non un motif repete")
print("  localement. Seul le modele « phase B », entraine avec augmentations,")
print("  en recupere une partie.")

---
# 7 · Le compromis, tracé

La force du filigrane règle tout : plus elle est haute, plus Ω se lit, plus
l'image se dégrade. Cette cellule balaie la plage et trace les deux courbes,
sur **votre** image et **votre** modèle.

La lecture utile est le point où la bit_acc décroche : c'est la limite
au-delà de laquelle on ne peut plus gagner en invisibilité.

In [ ]:
#@title  7 · Courbe force <-> fidelite <-> lisibilite  { display-mode: "form" }
POINTS = 7 #@param {type:"slider", min:4, max:12, step:1}
if "CM" not in globals():
    raise SystemExit("\n  ==> Chargez d'abord un couple (section 2).\n")

x0, origine = (x, "celle de la section 3") if "x" in globals() else image_de_travail()
depart = float(WAM.blender.scaling_w)
forces = np.linspace(0.02, max(0.16, depart), POINTS)
ps, accs = [], []
separateur("BALAYAGE DE LA FORCE DU FILIGRANE")
print(f"  {'force':>8}{'PSNR':>10}{'bit_acc':>10}{'verdict':>10}")
print("  " + "-" * 38)
for f in forces:
    WAM.blender.scaling_w = float(f)
    s = CM.embed(x0); m, i2 = s["imgs_w"], s["image_ids"]
    r = CM.verify(m, i2)[0]
    p, a = psnr(x0, m), 1 - r.distance / r.total
    ps.append(p); accs.append(a)
    print(f"  {f:>8.3f}{p:>10.2f}{a:>10.4f}"
          f"{('RENDU' if r.p_value < 1e-6 else 'aucun'):>10}")
WAM.blender.scaling_w = depart   # on rend le modele a son etat d'origine

fig, a1 = plt.subplots(figsize=(7.6, 3.6))
a1.plot(forces, ps, "o-", color="#2c7fb8", lw=2, label="PSNR (fidelite)")
a1.set_xlabel("force du filigrane"); a1.set_ylabel("PSNR (dB)", color="#2c7fb8")
a1.axhline(35, ls=":", color="#2c7fb8", lw=1)
a1.text(forces[0], 35.6, "35 dB : invisible a l'oeil", fontsize=8, color="#2c7fb8")
a2 = a1.twinx()
a2.plot(forces, accs, "s-", color="#c0392b", lw=2, label="bit_acc (lisibilite)")
a2.set_ylabel("bit_acc", color="#c0392b"); a2.set_ylim(0.45, 1.02)
a2.axhline(0.99, ls=":", color="#c0392b", lw=1)
a1.axvline(depart, ls="--", color="gray", lw=1)
a1.text(depart, min(ps), " valeur entrainee", fontsize=8, color="gray", rotation=90, va="bottom")
fig.suptitle(f"Le compromis, sur {origine}", fontsize=10)
plt.tight_layout(); plt.show()
print("\n  Les deux courbes vont en sens inverse : c'est le compromis central du")
print("  tatouage. Le bon reglage est le PSNR le plus haut qui garde bit_acc >= 0,99.")

---
# Pour aller plus loin

Ce carnet ne couvre que le couple **pixel**. Le reste du système est ailleurs :

- **`CipherMark-banc-essai-des-modeles.ipynb`** — les décodeurs génératifs
  conditionnés, la chaîne prompt→verdict, l'injection latente.
- **`CipherMark-presentation-des-resultats.ipynb`** — toutes les mesures du
  mémoire, dont les 300 verdicts sur 300 de la chaîne complète et les 25 000
  vérifications d'attribution sans une seule erreur.

Les données brutes sont dans `ciphermark/` sur le Drive, un dossier par étape
du pipeline, chacun avec son `LISEZ-MOI.md`.